In [0]:
-- monthly sales

CREATE OR REPLACE MATERIALIZED VIEW gold_monthly_sales
AS
SELECT
    d.year,
    d.month_num,
    d.month_name,
    DATE_TRUNC('month', f.sale_date) AS month_start,
    COUNT(*) AS transaction_count,
    SUM(f.sale_dollars) AS total_revenue,
    SUM(f.bottles_sold) AS total_bottles,
    SUM(f.volume_sold_liters) AS total_liters,
    SUM(f.profit) AS total_profit,
    AVG(f.sale_dollars) AS avg_transaction_value,
    COUNT(DISTINCT f.store_id) AS active_stores,
    COUNT(DISTINCT f.vendor_id) AS active_vendors
FROM ${source_catalog}.silver.fct_sales f
JOIN ${source_catalog}.silver.dim_dates d
    ON f.sale_date = d.date_day
GROUP BY d.year, d.month_num, d.month_name, DATE_TRUNC('month', f.sale_date)

In [0]:
-- day of week patterns
CREATE OR REPLACE MATERIALIZED VIEW gold_dow_patterns
AS
SELECT
    d.day_of_week,
    d.day_name,
    d.is_weekend,
    COUNT(*) AS transactions,
    SUM(f.sale_dollars) AS revenue,
    AVG(f.sale_dollars) AS avg_sale
FROM ${source_catalog}.silver.fct_sales f
JOIN ${source_catalog}.silver.dim_dates d
    ON f.sale_date = d.date_day
GROUP BY d.day_of_week, d.day_name, d.is_weekend

In [0]:
-- seasonal category performance

CREATE OR REPLACE MATERIALIZED VIEW gold_seasonal_categories
AS
SELECT
    d.month_num,
    d.month_name,
    p.category_name,
    COUNT(*) AS transactions,
    SUM(f.sale_dollars) AS revenue,
    SUM(f.bottles_sold) AS bottles,
    RANK() OVER (
        PARTITION BY d.month_num
        ORDER BY SUM(f.sale_dollars) DESC
    ) AS category_rank_in_month
FROM ${source_catalog}.silver.fct_sales f
JOIN ${source_catalog}.silver.dim_dates d
    ON f.sale_date = d.date_day
JOIN ${source_catalog}.silver.dim_products p
    ON f.product_id = p.product_id
GROUP BY d.month_num, d.month_name, p.category_name